In [ ]:
#@title Cell 0 — Setup (run me first)
# ============================================================
# Shared Infrastructure: company_sim.py (embedded)
# ============================================================
import json, os
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from statsmodels.stats.outliers_influence import variance_inflation_factor, OLSInfluence
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

COLORS = {
    'ols': '#2E5EA8',
    'truth': '#D4A017',
    'bias': '#C0392B',
    'repair': '#27AE60',
    'alt': '#7F8C8D',
}

_trap_responses = {}
_TRAP_LOG_PATH = '/content/trap_log.json'

def _load_trap_log():
    global _trap_responses
    if os.path.exists(_TRAP_LOG_PATH):
        try:
            with open(_TRAP_LOG_PATH, 'r') as f:
                _trap_responses = json.load(f)
        except (json.JSONDecodeError, IOError):
            _trap_responses = {}

def _save_trap_log():
    try:
        os.makedirs(os.path.dirname(_TRAP_LOG_PATH), exist_ok=True)
        with open(_TRAP_LOG_PATH, 'w') as f:
            json.dump(_trap_responses, f, indent=2)
    except IOError:
        pass

def record_trap_response(notebook_id, question_id, response):
    key = f"{notebook_id}_{question_id}"
    _trap_responses[key] = {"response": response, "timestamp": datetime.now().isoformat()}
    _save_trap_log()

def get_trap_response(notebook_id, question_id):
    key = f"{notebook_id}_{question_id}"
    entry = _trap_responses.get(key)
    return entry["response"] if entry else None

def check_gate(notebook_id, question_id):
    return f"{notebook_id}_{question_id}" in _trap_responses

def get_all_responses():
    return dict(_trap_responses)

def create_trap_widget(notebook_id, question_id, question_text, options):
    label = widgets.HTML(f"<b>{question_text}</b>")
    radio = widgets.RadioButtons(options=options, value=None, layout=widgets.Layout(width='auto'))
    submit = widgets.Button(description="Submit", button_style='primary')
    output = widgets.Output()
    def on_submit(btn):
        with output:
            output.clear_output()
            if radio.value is None:
                print("Please select an option before submitting.")
                return
            record_trap_response(notebook_id, question_id, radio.value)
            print(f"Response recorded: {radio.value}")
            submit.disabled = True
            radio.disabled = True
    submit.on_click(on_submit)
    existing = get_trap_response(notebook_id, question_id)
    if existing is not None:
        try:
            radio.value = existing
            radio.disabled = True
            submit.disabled = True
        except Exception:
            pass
    return widgets.VBox([label, radio, submit, output])

_load_trap_log()

class CompanySimulator:
    def __init__(self, endogeneity_strength=20, heteroscedasticity_strength=0.5,
                 nonlinearity=True, bad_control_strength=0.1, noise_sigma=1.0):
        self.endogeneity_strength = endogeneity_strength
        self.heteroscedasticity_strength = heteroscedasticity_strength
        self.nonlinearity = nonlinearity
        self.bad_control_strength = bad_control_strength
        self.noise_sigma = noise_sigma
        self.beta_0, self.beta_1, self.beta_2, self.beta_U = 50, 8, 3, 2
        self.staff_loading = 5
    def generate(self, n=500, seed=42):
        rng = np.random.default_rng(seed)
        U = rng.standard_normal(n)
        eta1, eta2, eta3 = [rng.normal(0, self.noise_sigma, n) for _ in range(3)]
        X1 = self.endogeneity_strength * U + eta1
        if self.nonlinearity: X1 = np.abs(X1) + 0.01
        X2 = self.staff_loading * U + eta2
        eps = rng.normal(0, np.sqrt(np.abs(self.heteroscedasticity_strength * X1)))
        Y = self.beta_0 + (self.beta_1 * np.log(X1) if self.nonlinearity else self.beta_1 * X1) + self.beta_2 * X2 + self.beta_U * U + eps
        X3 = self.bad_control_strength * Y + eta3
        return pd.DataFrame({'revenue': Y, 'ad_spend': X1, 'staff_count': X2, 'satisfaction': X3})
    def truth(self, n=500, seed=42):
        rng = np.random.default_rng(seed)
        U = rng.standard_normal(n)
        eta1, eta2, eta3 = [rng.normal(0, self.noise_sigma, n) for _ in range(3)]
        X1 = self.endogeneity_strength * U + eta1
        if self.nonlinearity: X1 = np.abs(X1) + 0.01
        X2 = self.staff_loading * U + eta2
        eps = rng.normal(0, np.sqrt(np.abs(self.heteroscedasticity_strength * X1)))
        Y = self.beta_0 + (self.beta_1 * np.log(X1) if self.nonlinearity else self.beta_1 * X1) + self.beta_2 * X2 + self.beta_U * U + eps
        X3 = self.bad_control_strength * Y + eta3
        df = pd.DataFrame({'revenue': Y, 'ad_spend': X1, 'staff_count': X2, 'satisfaction': X3, 'demand_U': U})
        return df, {'beta_0': self.beta_0, 'beta_1': self.beta_1, 'beta_2': self.beta_2, 'beta_U': self.beta_U}
    def set_heteroscedasticity(self, s): self.heteroscedasticity_strength = s
    def set_endogeneity(self, s): self.endogeneity_strength = s
    def set_nonlinearity(self, c): self.nonlinearity = bool(c)

# Generate NovaMart data (linear version for clean FWL demo)
sim = CompanySimulator(nonlinearity=False, heteroscedasticity_strength=0.0, noise_sigma=2.0)
data = sim.generate(n=500, seed=42)

print('\u2705 Setup complete.')
print(f'NovaMart data: {len(data)} observations')
print(f'Variables: {list(data.columns)}')

# Notebook 1.5: What "Controlling For" Actually Means

**20 minutes** | Optional deep dive

---

## The Question

In Notebook 1, you added `staff_count` as a control variable. The coefficient on `ad_spend` changed.

But what did that *do*, mechanically? When we say "controlling for staff count," what is the regression actually computing?

Most people have an intuition: "it holds staff count constant and looks at the remaining relationship." That intuition is partially right — but it misses something important about what "holding constant" really means in practice.

Let's find out.

In [ ]:
# ============================================================
# The Trap: What does "controlling for" mean?
# ============================================================

# Run the multiple regression
X_multi = sm.add_constant(data[['ad_spend', 'staff_count']])
model_multi = sm.OLS(data['revenue'], X_multi).fit()

coef_ad = model_multi.params['ad_spend']

print('Multiple Regression: revenue ~ ad_spend + staff_count')
print('='*60)
print(model_multi.summary().tables[1])
print(f'\nCoefficient on ad_spend: {coef_ad:.4f}')
print()

# Trap widget
trap = create_trap_widget(
    'notebook_1b', 'q1',
    'What does the coefficient on ad_spend mean in this multiple regression?',
    [
        f'A: "Holding staff constant, each unit of ad spend increases revenue by {coef_ad:.1f}"',
        f'B: "After residualizing both sides on staff, the remaining relationship has slope {coef_ad:.1f}"',
        'C: "Both A and B are correct and mean the same thing"',
        'D: "Neither is correct"',
    ]
)
display(trap)

In [ ]:
# ============================================================
# The Reveal: Frisch-Waugh-Lovell Theorem
# ============================================================

if not check_gate('notebook_1b', 'q1'):
    print('\u26a0\ufe0f Please answer the question in the previous cell before continuing.')
else:
    student_answer = get_trap_response('notebook_1b', 'q1')
    print(f'Your answer: {student_answer}')
    print()
    
    if 'B:' in str(student_answer):
        print('\u2705 Correct! Option B describes what regression actually does.')
    elif 'C:' in str(student_answer):
        print('\u274c Close, but A and B are NOT the same thing.')
        print('"Holding constant" is a convenient shorthand that hides the mechanism.')
    else:
        print('\u274c The correct answer is B.')
    
    print()
    print('='*70)
    print('THE FRISCH-WAUGH-LOVELL (FWL) THEOREM')
    print('='*70)
    print()
    print('Multiple regression doesn\'t "hold variables constant."')
    print('It strips out what the controls explain about BOTH sides,')
    print('then measures what\'s left.')
    print()
    print('FWL Procedure (manual):')
    print('  Step 1: Regress revenue on staff_count \u2192 save residuals')
    print('  Step 2: Regress ad_spend on staff_count \u2192 save residuals')
    print('  Step 3: Regress revenue_residuals on ad_spend_residuals')
    print('  The coefficient from Step 3 = the multiple regression coefficient. Exactly.')
    print()
    
    # Step 1: revenue ~ staff_count
    X_staff = sm.add_constant(data[['staff_count']])
    model_rev_staff = sm.OLS(data['revenue'], X_staff).fit()
    rev_resid = model_rev_staff.resid
    
    # Step 2: ad_spend ~ staff_count
    model_ad_staff = sm.OLS(data['ad_spend'], X_staff).fit()
    ad_resid = model_ad_staff.resid
    
    # Step 3: rev_resid ~ ad_resid
    X_resid = sm.add_constant(ad_resid)
    model_fwl = sm.OLS(rev_resid, X_resid).fit()
    fwl_coef = model_fwl.params.iloc[1]  # slope on ad_resid
    
    print(f'Multiple regression coefficient: {coef_ad:.14f}')
    print(f'FWL procedure coefficient:       {fwl_coef:.14f}')
    print(f'Match to 14 decimal places:      {abs(coef_ad - fwl_coef) < 1e-10}')
    print()
    
    # Side-by-side plots
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Left: original scatterplot
    ax = axes[0]
    ax.scatter(data['ad_spend'], data['revenue'], alpha=0.4, s=15,
               color=COLORS['ols'], marker='o')
    # Simple regression line
    X_simple = sm.add_constant(data[['ad_spend']])
    model_simple = sm.OLS(data['revenue'], X_simple).fit()
    x_range = np.linspace(data['ad_spend'].min(), data['ad_spend'].max(), 100)
    ax.plot(x_range, model_simple.predict(sm.add_constant(x_range)),
            color=COLORS['bias'], linewidth=2, linestyle=':',
            label=f'Simple OLS: {model_simple.params.iloc[1]:.2f}')
    ax.set_xlabel('Ad Spend', fontsize=12)
    ax.set_ylabel('Revenue', fontsize=12)
    ax.set_title('Original Data', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    
    # Right: partial regression plot (residuals)
    ax = axes[1]
    ax.scatter(ad_resid, rev_resid, alpha=0.4, s=15,
               color=COLORS['repair'], marker='s')
    x_range_r = np.linspace(ad_resid.min(), ad_resid.max(), 100)
    ax.plot(x_range_r, model_fwl.predict(sm.add_constant(x_range_r)),
            color=COLORS['repair'], linewidth=2,
            label=f'FWL slope: {fwl_coef:.2f}')
    ax.set_xlabel('Ad Spend residuals (staff removed)', fontsize=12)
    ax.set_ylabel('Revenue residuals (staff removed)', fontsize=12)
    ax.set_title('Partial Regression Plot (FWL)', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    
    fig.tight_layout()
    plt.show()
    
    print('Left: the raw relationship (confounded by staff).')
    print('Right: after stripping out staff from BOTH sides, the partial relationship.')
    print(f'The partial regression slope ({fwl_coef:.4f}) IS the multiple regression coefficient.')

In [ ]:
# ============================================================
# Interactive: Checkbox list of control variables
# ============================================================

# Use all available controls
all_controls = ['staff_count', 'satisfaction']

checkboxes = {var: widgets.Checkbox(value=(var == 'staff_count'), description=var,
                                     layout=widgets.Layout(width='200px'))
              for var in all_controls}

out_fwl = widgets.Output()

def update_fwl(change=None):
    with out_fwl:
        clear_output(wait=True)
        
        selected = [v for v, cb in checkboxes.items() if cb.value]
        
        if not selected:
            # Simple regression (no controls)
            X_full = sm.add_constant(data[['ad_spend']])
            model_full = sm.OLS(data['revenue'], X_full).fit()
            full_coef = model_full.params['ad_spend']
            
            fig, ax = plt.subplots(figsize=(7, 5))
            ax.scatter(data['ad_spend'], data['revenue'], alpha=0.4, s=15, color=COLORS['ols'])
            x_r = np.linspace(data['ad_spend'].min(), data['ad_spend'].max(), 100)
            ax.plot(x_r, model_full.predict(sm.add_constant(x_r)),
                    color=COLORS['ols'], linewidth=2)
            ax.set_xlabel('Ad Spend'); ax.set_ylabel('Revenue')
            ax.set_title(f'No controls | Coefficient: {full_coef:.4f}', fontsize=13, fontweight='bold')
            fig.tight_layout()
            plt.show()
            
            print(f'No controls selected. Simple regression coefficient: {full_coef:.4f}')
            return
        
        # Full multiple regression
        X_full = sm.add_constant(data[['ad_spend'] + selected])
        model_full = sm.OLS(data['revenue'], X_full).fit()
        full_coef = model_full.params['ad_spend']
        
        # FWL procedure
        X_controls = sm.add_constant(data[selected])
        rev_resid = sm.OLS(data['revenue'], X_controls).fit().resid
        ad_resid = sm.OLS(data['ad_spend'], X_controls).fit().resid
        fwl_model = sm.OLS(rev_resid, sm.add_constant(ad_resid)).fit()
        fwl_coef = fwl_model.params.iloc[1]
        
        match = abs(full_coef - fwl_coef) < 1e-10
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Left: full regression summary
        ax = axes[0]
        ax.axis('off')
        controls_str = ' + '.join(selected)
        text = f'revenue ~ ad_spend + {controls_str}\n\n'
        text += f'Coefficient on ad_spend: {full_coef:.6f}\n\n'
        text += 'All coefficients:\n'
        for name, val in model_full.params.items():
            if name == 'const': continue
            text += f'  {name}: {val:.4f}\n'
        ax.text(0.1, 0.5, text, transform=ax.transAxes, fontsize=12,
                verticalalignment='center', fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        ax.set_title('Full Multiple Regression', fontsize=13, fontweight='bold')
        
        # Right: partial regression plot
        ax = axes[1]
        has_bad_control = 'satisfaction' in selected
        color = COLORS['bias'] if has_bad_control else COLORS['repair']
        ax.scatter(ad_resid, rev_resid, alpha=0.4, s=15, color=color,
                   marker='s' if not has_bad_control else '^')
        x_r = np.linspace(ad_resid.min(), ad_resid.max(), 100)
        ax.plot(x_r, fwl_model.predict(sm.add_constant(x_r)),
                color=color, linewidth=2,
                label=f'FWL slope: {fwl_coef:.4f}')
        ax.set_xlabel(f'Ad Spend residuals ({controls_str} removed)', fontsize=11)
        ax.set_ylabel(f'Revenue residuals ({controls_str} removed)', fontsize=11)
        title = f'Partial Regression Plot | FWL = {fwl_coef:.4f}'
        if has_bad_control:
            title += ' [BAD CONTROL!]'
        ax.set_title(title, fontsize=13, fontweight='bold',
                     color=COLORS['bias'] if has_bad_control else 'black')
        ax.legend(fontsize=10)
        
        fig.tight_layout()
        plt.show()
        
        # Verification table
        match_icon = '\u2705' if match else '\u274c'
        print(f'Controls: {controls_str}')
        print(f'Full regression coefficient: {full_coef:.14f}')
        print(f'FWL procedure coefficient:   {fwl_coef:.14f}')
        print(f'Match: {match_icon} (difference: {abs(full_coef - fwl_coef):.2e})')
        
        if has_bad_control:
            print()
            print('\u26a0\ufe0f WARNING: satisfaction (X\u2083) is a BAD CONTROL!')
            print('  It is caused by revenue (post-treatment). Including it as a')
            print('  control strips out variation in revenue that IS caused by ad_spend,')
            print('  distorting the partial regression plot and biasing the coefficient.')

print('Toggle control variables to see FWL in action.')
print('The FWL coefficient always matches the full regression coefficient.\n')

checkbox_box = widgets.HBox(list(checkboxes.values()))
display(checkbox_box, out_fwl)

for cb in checkboxes.values():
    cb.observe(update_fwl, names='value')
update_fwl()

In [ ]:
# ============================================================
# Partial Regression Plot Grid
# ============================================================

predictors = ['ad_spend', 'staff_count', 'satisfaction']
pred_labels = ['Ad Spend', 'Staff Count', 'Satisfaction']

# Full model with all predictors
X_all = sm.add_constant(data[predictors])
model_all = sm.OLS(data['revenue'], X_all).fit()

out_select = widgets.Output()

dropdown = widgets.Dropdown(
    options=['(highlight all)', 'ad_spend', 'staff_count', 'satisfaction'],
    value='(highlight all)',
    description='Highlight:',
    style={'description_width': 'initial'}
)

def update_grid(change=None):
    with out_select:
        clear_output(wait=True)
        
        highlight = dropdown.value
        
        fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
        
        for idx, (pred, label) in enumerate(zip(predictors, pred_labels)):
            ax = axes[idx]
            
            # FWL: residualize both Y and this predictor on all OTHER predictors
            others = [p for p in predictors if p != pred]
            X_others = sm.add_constant(data[others])
            
            y_resid = sm.OLS(data['revenue'], X_others).fit().resid
            x_resid = sm.OLS(data[pred], X_others).fit().resid
            
            fwl = sm.OLS(y_resid, sm.add_constant(x_resid)).fit()
            fwl_coef = fwl.params.iloc[1]
            full_coef = model_all.params[pred]
            
            # Color based on highlight
            is_highlighted = (highlight == '(highlight all)' or highlight == pred)
            alpha = 0.5 if is_highlighted else 0.08
            
            is_bad = (pred == 'satisfaction')
            base_color = COLORS['bias'] if is_bad else COLORS['ols']
            marker = '^' if is_bad else 'o'
            
            ax.scatter(x_resid, y_resid, alpha=alpha, s=12, color=base_color, marker=marker)
            
            if is_highlighted:
                x_r = np.linspace(x_resid.min(), x_resid.max(), 100)
                ax.plot(x_r, fwl.predict(sm.add_constant(x_r)),
                        color=base_color, linewidth=2)
            
            ax.set_xlabel(f'{label} (partial)', fontsize=11)
            ax.set_ylabel('Revenue (partial)' if idx == 0 else '', fontsize=11)
            title = f'{label}: {fwl_coef:.4f}'
            if is_bad:
                title += ' [bad control]'
            ax.set_title(title, fontsize=12, fontweight='bold',
                         color=COLORS['bias'] if is_bad else 'black')
        
        fig.suptitle('Partial Regression Plots (FWL for each predictor)',
                     fontsize=14, fontweight='bold', y=1.02)
        fig.tight_layout()
        plt.show()
        
        print('Each plot shows the relationship between a predictor and revenue')
        print('after removing the effect of all other predictors from BOTH variables.')
        print('The slope in each partial plot = the coefficient in the full regression.')

print('Partial Regression Plots')
print('Select a variable to highlight it across all plots.\n')
display(dropdown, out_select)

dropdown.observe(update_grid, names='value')
update_grid()

---

## Connections

**Back to Notebook 1 (Why Your Coefficient Is Wrong):**
Now you know *mechanically* why adding `staff_count` as a control changed the coefficient on `ad_spend`. The FWL theorem shows that the multiple regression coefficient measures the relationship between the *parts of ad_spend and revenue that staff_count can't explain*. When staff_count absorbs some of the variation that was previously attributed to ad_spend (because both are driven by demand), the partial relationship changes.

This is why omitted variable bias works the way it does: if a confounder drives variation in both X and Y, failing to residualize on it leaves that confounded variation in the partial relationship.

**Forward to Notebook 2 (Why Your Uncertainty Is Wrong):**
The FWL theorem also explains why adding controls affects standard errors. The variance of the coefficient depends on the residual variation in the predictor *after removing controls*. If controls absorb a lot of the variation in X, the residuals have less variation, and the standard errors increase.

---

<div style='background: #1a1a2e; color: white; padding: 20px; border-radius: 10px; margin: 20px 0;'>
    <h3 style='color: #D4A017; margin-top: 0;'>Return to the main sequence</h3>
    <p>Continue with Notebook 2: Why Your Uncertainty Is Wrong</p>
    <p><em>"Your coefficient might be right but your margin of error is wrong."</em></p>
</div>